In [2]:
import pandas as pd

df = pd.read_csv('data.csv', encoding='cp949', low_memory=False)

cols = ['영업상태명', '업태구분명', '인허가일자', '폐업일자', '도로명주소']

df_sub = df[cols]

df_sub.head()

,영업상태명,업태구분명,인허가일자,폐업일자,도로명주소
0,폐업,커피숍,2026-06-18,2026-06-20,"서울특별시 종로구 동숭길 122, 서울문화재단 대학로 (동숭동)"
1,영업/정상,커피숍,2026-06-18,NaN,"서울특별시 종로구 창의문로3길 1, 2층 (부암동)"
2,영업/정상,기타 휴게음식점,2026-06-04,NaN,"서울특별시 종로구 종로 335, 원풍빌딩 지하1층 (창신동)"
3,영업/정상,커피숍,2023-12-12,NaN,"서울특별시 종로구 창경궁로35길 17, 1층 (혜화동)"
4,영업/정상,기타 휴게음식점,2026-06-23,NaN,"서울특별시 종로구 종로 78, 미려빌딩 지하1층 (종로2가)"


In [3]:
df_coffee = df_sub[df_sub['업태구분명'] == '커피숍']

df_seoul = df_coffee[df_coffee['도로명주소'].str.contains('서울특별시', na=False)]

df_seoul

,영업상태명,업태구분명,인허가일자,폐업일자,도로명주소
0,폐업,커피숍,2026-06-18,2026-06-20,"서울특별시 종로구 동숭길 122, 서울문화재단 대학로 (동숭동)"
1,영업/정상,커피숍,2026-06-18,NaN,"서울특별시 종로구 창의문로3길 1, 2층 (부암동)"
3,영업/정상,커피숍,2023-12-12,NaN,"서울특별시 종로구 창경궁로35길 17, 1층 (혜화동)"
10,영업/정상,커피숍,2026-06-10,NaN,"서울특별시 종로구 대학로 83, 1층 일부호 (연건동)"
13,영업/정상,커피숍,2026-06-30,NaN,"서울특별시 종로구 자하문로9길 34-13, 3층 (누하동)"
...,...,...,...,...,...
146175,폐업,커피숍,2017-07-11,2022-11-07,"서울특별시 강동구 성내로6길 14, 1층 (성내동)"
146177,영업/정상,커피숍,2021-01-12,NaN,"서울특별시 강동구 성내로 7, 1층 101호 (성내동)"
146178,폐업,커피숍,2015-04-01,2016-05-03,서울특별시 강동구 고덕로25길 13-19 (암사동)
146179,폐업,커피숍,2018-05-18,2024-07-24,"서울특별시 강동구 진황도로 110, 1층 102호 (길동)"


In [8]:
# 1. '도로명주소'에서 '구' 이름 추출하기 (예: '서울특별시 강남구 ...' -> '강남구')
df_seoul['구'] = df_seoul['도로명주소'].str.extract(r'([가-힣]+구)')

# 2. 인허가일자와 폐업일자를 날짜 형식(datetime)으로 변환
df_seoul['인허가일자'] = pd.to_datetime(df_seoul['인허가일자'].astype(str), errors='coerce')
df_seoul['폐업일자'] = pd.to_datetime(df_seoul['폐업일자'].astype(str), errors='coerce')

# 3. 개업연도, 폐업연도 컬럼 따로 만들기 (년도만 쏙 뽑아내기)
df_seoul['개업연도'] = df_seoul['인허가일자'].dt.year
df_seoul['폐업연도'] = df_seoul['폐업일자'].dt.year

# 1. 최근 3년(2023~2025)
df_open = df_seoul[(df_seoul['개업연도'] >= 2023) & (df_seoul['개업연도'] <= 2025)]
df_close = df_seoul[(df_seoul['폐업연도'] >= 2023) & (df_seoul['폐업연도'] <= 2025)]

# 2. groupby로 구별 개수 세기
open_by_gu = df_open.groupby('구').size()
close_by_gu = df_close.groupby('구').size()

# 3. 새로운 데이터프레임(df_result)에 컬럼 하나씩 담아 정리하기
df_result = pd.DataFrame()
df_result['개업수'] = open_by_gu
df_result['폐업수'] = close_by_gu

# 4. 순감소수(폐업 - 개업) 계산
df_result['순감소수'] = df_result['폐업수'] - df_result['개업수']

# 5. 폐업이 개업보다 많은(순감소수가 큰) 순서대로 정렬
df_result = df_result.sort_values(by='순감소수', ascending=False)
# 6. 개폐업비율 계산 (폐업수 / 개업수)
df_result['개폐업비율'] = (df_result['폐업수'] / df_result['개업수']).round(2)

df_result

,개업수,폐업수,순감소수,개폐업비율
구,,,,
강서구,455,590,135,1.30
마포구,464,560,96,1.21
영등포구,276,345,69,1.25
은평구,124,190,66,1.53
양천구,175,233,58,1.33
중랑구,121,172,51,1.42
성북구,210,260,50,1.24
동작구,151,197,46,1.30
관악구,267,309,42,1.16
